In [ ]:
# Monte Carlo Method

import numpy as np
import gymnasium as gym

env = gym.make("FrozenLake-v1")


def monte_carlo(env, episodes=500, gamma=0.9):
    returns = {}
    V = np.zeros(env.observation_space.n)

    for ep in range(episodes):
        episode = []

        state = env.reset()[0] if isinstance(env.reset(), tuple) else env.reset()
        done = False

        while not done:
            action = env.action_space.sample()

            step = env.step(action)

            if len(step) == 5:  # gymnasium
                next_state, reward, terminated, truncated, _ = step
                done = terminated or truncated
            else:  # gym
                next_state, reward, done, _ = step

            episode.append((state, reward))
            state = next_state

        # Compute returns
        G = 0

        for state, reward in reversed(episode):
            G = gamma * G + reward

            if state not in returns:
                returns[state] = []

            returns[state].append(G)
            V[state] = np.mean(returns[state])

    return V


print("\n===== MONTE CARLO OUTPUT =====")

V_mc = monte_carlo(env)

for i, v in enumerate(V_mc):
    print(f"State {i}: {v:.4f}")

print("\nGrid View:")
print(V_mc.reshape(4, 4))

In [ ]:
# TD Learning

import numpy as np
import gymnasium as gym

env = gym.make("FrozenLake-v1")


def td_learning(env, episodes=500, alpha=0.1, gamma=0.9):
    V = np.zeros(env.observation_space.n)

    for ep in range(episodes):
        state = env.reset()[0]
        done = False

        while not done:
            action = env.action_space.sample()

            step = env.step(action)

            if len(step) == 5:  # gymnasium
                next_state, reward, terminated, truncated, _ = step
                done = terminated or truncated
            else:  # gym
                next_state, reward, done, _ = step

            V[state] += alpha * (
                reward + gamma * V[next_state] - V[state]
            )

            state = next_state

    return V


print("\n===== TD LEARNING OUTPUT =====")

V_td = td_learning(env)

for i, v in enumerate(V_td):
    print(f"State {i}: {v:.4f}")

print("\nGrid View:")
print(V_td.reshape(4, 4))

In [ ]:
# Q-Learning

import numpy as np
import gymnasium as gym

env = gym.make("FrozenLake-v1")


def q_learning(env, episodes=2000, alpha=0.1, gamma=0.9, epsilon=0.1):
    Q = np.zeros((env.observation_space.n, env.action_space.n))

    for ep in range(episodes):
        state = env.reset()[0]
        done = False

        while not done:
            # Epsilon-greedy
            if np.random.rand() < epsilon:
                action = env.action_space.sample()
            else:
                action = np.argmax(Q[state])

            step = env.step(action)

            if len(step) == 5:  # gymnasium
                next_state, reward, terminated, truncated, _ = step
                done = terminated or truncated
            else:  # gym
                next_state, reward, done, _ = step

            Q[state, action] += alpha * (
                reward
                + gamma * np.max(Q[next_state])
                - Q[state, action]
            )

            state = next_state

    return Q


print("\n===== Q-LEARNING OUTPUT =====")

Q = q_learning(env)

print("\nQ-table:")
print(Q)

# Derived policy
policy = np.argmax(Q, axis=1)

print("\nDerived Policy (State → Action):")
for i, a in enumerate(policy):
    print(f"State {i} → Action {a}")

print("\nPolicy Grid:")
print(policy.reshape(4, 4))

print("\nValue Grid:")
print(np.max(Q, axis=1).reshape(4, 4))